# Cell Analysis 

## Concepts and work flow : 

Using the developed tools we will do the following for one cell : 
1. load data 
2. segment data and process number of blocks 
3. decide on KPIs to extract 
4. Extract KPIs for each cycle 
5. visualize KPIs for the cell


## Imports and Data preparation and processing 

In [12]:
import sys
import os
import yaml
import pandas as pd 


# Add src/ to sys.path so you can import batterydata
sys.path.append(os.path.abspath('../src'))

from batterydata.parsers.basytec_reader import BasyTecReader
from batterydata.pipeline.preprocessor import Preprocessor

In [2]:


with open('../local_config.yaml', 'r') as f:
    config = yaml.safe_load(f)
file_path = config['basytec_sample_file']


# Load data
reader = BasyTecReader(filepath=file_path, sample_id="demo")
result = reader.read()
df = result['data']




In [ ]:
# Instantiate Preprocessor and process
preproc = Preprocessor(df, cell_capacity_ah=1.3)
seg, doe, blocks = preproc.process_with_blocks(min_block_len=4, max_block_len=4, min_repeats=2)
tables = preproc.export_tables()
segmented_df = tables["segmented_data"]
doe_table = tables["doe_table"]
metadata_table = tables["metadata"]

In [16]:
try:
    pd.testing.assert_frame_equal(segmented_df, seg)
    print("DataFrames are exactly equal")
except AssertionError as e:
    print("DataFrames differ:", e)



DataFrames are exactly equal


In [4]:
from typing import List, Dict

def pretty_print_blocks(blocks: List[Dict[str, object]]) -> None:
    """Print a readable summary of detected blocks.

    Parameters
    ----------
    blocks : list of dict
        Output from detect_repeating_blocks_with_steps() or find_repeating_blocks().
    """
    for b in blocks:
        step = b['start']
        if b['count'] > 1:
            print(f"Step {step}: {b['count']} x ({', '.join(b['block'])})")
        else:
            print(f"Step {step}: {', '.join(b['block'])}")


In [17]:
pretty_print_blocks(blocks)

Step 0: DIS_1.00_V3.0_noCV
Step 1: OCV_600
Step 2: 200 x (CH_1.00_V4.2_CV, OCV_10, DIS_1.00_V3.0_noCV, OCV_10)
Step 802: DIS_1.00_V3.0_CV
Step 803: OCV_10
Step 804: CH_0.05_V4.2_noCV
Step 805: OCV_10
Step 806: DIS_0.05_V3.0_CV
Step 807: 160 x (OCV_10, CH_1.00_V4.2_CV, OCV_10, DIS_1.00_V3.0_noCV)
Step 1447: OCV_10
Step 1448: CH_1.00_V3.8_CV
Step 1449: 35 x (CH_1.00_V4.2_CV, OCV_10, DIS_1.00_V3.0_noCV, OCV_10)
Step 1589: CH_1.00_V4.2_CV
Step 1590: OCV_10
Step 1591: DIS_1.00_V3.9_CV


This indicates that the cell had :
1. Discharge initial step 
2. Rest for 600 seconds
3. 200 x 
    1. charge CCCV
    2. rest 10 seconds
    3. Discharge CC 
    4. rest 10 seconds 
4. Discharge 
5. qOCV step : 
    1. Charge CCCV C/20
    2. rest 10 seconds 
    3. Discharge CC C/20


## Defining KPIs 



Lets define the following KPIs to be calculated for this test procedure. 

First of all for each cycle calculate 

<ol type='a'>
<li> charge capacity</li>
<li> discharge capcity </li>
<li> Energy charge </li>
<li> Energy discharge </li>
<li> Culombic Efficiency C/E % </li>
<li> Energy Efficiency E/E % </li>
<li> Capacity retention (Qi / Q1) </li>
<li> Total charge time </li>
<li> CV step duration</li>
<li> CC step duration </li>
<li> CV duration / total charge time </li>

</ol>
